## Download Modules

In [14]:
%pip install -q --upgrade ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [30]:
%pip install -q transformers datasets pandas tqdm numpy torch

Note: you may need to restart the kernel to use updated packages.


## Import Modules

In [31]:
from typing import List, Any
from datasets import DatasetDict, Dataset
from transformers import (BartForConditionalGeneration, BartTokenizer, Trainer, TrainingArguments,
                          BartConfig, GenerationConfig, DataCollatorForSeq2Seq)
import pandas as pd
import math
from tqdm import tqdm
import numpy as np

In [17]:
import os
print("Current Working Directory:", os.getcwd())

Current Working Directory: /home/ubuntu/CiteBART/acl_200_global_rerun


In [18]:
custom_model_name = "citeBART_global_ACL200_rerun"
checkpoints_location = f"./checkpoints/{custom_model_name}"
model_save_location = f"./models/{custom_model_name}"

# Ensure directories exist
os.makedirs(checkpoints_location, exist_ok=True)
os.makedirs(model_save_location, exist_ok=True)

## Defining Functions

### Preprocessing Dataset

In [19]:
# Okay
def read_dataset():
    train_df = pd.read_csv(train_dataset_path)
    train_set = []

    for _, i in train_df.iterrows():
        temp_citing_title = i['citing_title']
        temp_citing_abstract = i['citing_abstract']
        temp_masked_context = i['masked_cit_context'].replace("OTHERCIT", "")  # "Fill the mask with an appropriate citation: " +

        temp_train_input = temp_citing_title + " </s> " + temp_citing_abstract + " </s> " + temp_masked_context

        temp_dict = {"masked_cit_context": temp_train_input, 
                     "masked_token_target": i['masked_token_target']}

        train_set.append(temp_dict)

    eval_df = pd.read_csv(eval_dataset_path)
    eval_set = []

    for _, i in eval_df.iterrows():
        temp_citing_title = i['citing_title']
        temp_citing_abstract = i['citing_abstract']
        temp_masked_context = i['masked_cit_context'].replace("OTHERCIT", "")  # "Fill the mask with an appropriate citation: " +

        temp_eval_input = temp_citing_title + " </s> " + temp_citing_abstract + " </s> " + temp_masked_context

        temp_dict = {"masked_cit_context": temp_eval_input,
                     "masked_token_target": i['masked_token_target']}

        eval_set.append(temp_dict)

    return train_set, eval_set

In [20]:
# Preprocessing function (Okay)
def preprocess_function(examples):
    inputs = [example.replace("<mask>", "<extra_id_0>", 1).replace("<mask>", "").replace("<extra_id_0>", "<mask>")
              for example in examples["masked_cit_context"]]
    targets = [example for example in examples["masked_token_target"]]

    model_inputs = tokenizer(inputs, max_length=max_token_limit, truncation=True, padding="max_length")
    labels = tokenizer(targets, max_length=max_token_limit, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

### Evaluation Functions

In [21]:
# Okay
def add_spaces_after_commas(text: str) -> str:
    return ', '.join(part.strip() for part in text.split(','))  #  NEW!!!!

# Okay
def fill_mask(sentence, num_predictions):

    # Read CSV and extract relevant column as a list (Optimized)
    all_cit_list = pd.read_csv(all_citations_path)['citation_items'].tolist()

    # Tokenize input
    sentence = sentence.replace("<mask>", "<extra_id_0>", 1).replace("<mask>", "").replace("<extra_id_0>", "<mask>")
    input_ids = tokenizer.encode(sentence, return_tensors="pt", max_length=max_token_limit, truncation=True, padding="max_length").to("cuda")

    # Generate outputs
    outputs = model.generate(input_ids, generation_config=cit_generation_config)

    # Decode outputs efficiently
    predictions = [add_spaces_after_commas(tokenizer.decode(output, skip_special_tokens=True).strip()) for output in outputs]

    # Get unique predictions
    unique_predictions: List[Any] = list(dict.fromkeys(predictions))  # Remove duplicates while preserving order

    # Print the top n predictions
    # for i, pred in enumerate(unique_predictions, num_predictions):
    #     print(f"Prediction {i}: {pred} \n\n")

    # Ensure at least `num_predictions` predictions
    if len(unique_predictions) < num_predictions:
        unique_predictions.extend([unique_predictions[-1]] * (num_predictions - len(unique_predictions)))

    return unique_predictions[:num_predictions]

In [22]:
# Okay
def compare_pred_with_correct_value(predictions, ground_truth, k=10):
    """Compute Recall@k, MRR, and NDCG for a single ground truth citation."""

    recall_at_k = 0
    reciprocal_rank = 0
    ndcg = 0

    #print(f"Predictions: {predictions}")
    #print(f"Number of Predictions: {len(predictions)})
    #print(f"Ground Truth: {ground_truth}")

    # Normalize ground truth
    ground_truth = ground_truth.replace(" and ", " ").replace(" et al.,", "").replace(",", "").strip()
    truth_tokens = ground_truth.split()

    #print(f"Truth Tokens: {truth_tokens}")

    # Ensure at least two tokens for comparison
    if len(truth_tokens) < 2:
        return recall_at_k, reciprocal_rank, ndcg # No valid NDCG if no valid ground truth
    
    # Check if the ground truth appears in the top-k predictions
    for p_idx, prediction in enumerate(predictions[:k]):  # Consider only top-k predictions
        if all(token in prediction for token in truth_tokens):  # Check if any token matches
            recall_at_k = 1  # Since there's only one correct answer
            reciprocal_rank = 1 / (p_idx + 1)
            ndcg = 1 / np.log2(p_idx + 2)  # Compute DCG for rank position
            break  # No need to check further once we find the match
        
    #print(f"Recall@k: {recall_at_k}, RR: {reciprocal_rank}, NDCG: {ndcg}")

    return recall_at_k, reciprocal_rank, ndcg

In [23]:
# Okay
def calc_eval_metrics(val_dataset, k=10):
    recall_list = []
    reciprocal_rank_list = []
    ndcg_list = []
    
    write_log("=== Evaluation Started ===")  # Log start of evaluation

    for e in tqdm(val_dataset):
        masked_cit_context = e["masked_cit_context"]
        target_token = e["masked_token_target"]

        temp_predictions = fill_mask(masked_cit_context, k)  # Get top-k predictions
        
        # Compute Recall@k, MRR, and NDCG@k
        recall_at_k, temp_reciprocal_rank, temp_ndcg = compare_pred_with_correct_value(temp_predictions, target_token, k)
        
        recall_list.append(recall_at_k)
        reciprocal_rank_list.append(temp_reciprocal_rank)
        ndcg_list.append(temp_ndcg)
    
    # Compute Mean Metrics
    mean_recall_at_k = np.mean(recall_list) if recall_list else 0
    mean_reciprocal_rank_at_k = np.mean(reciprocal_rank_list) if reciprocal_rank_list else 0
    mean_ndcg_at_k = np.mean(ndcg_list) if ndcg_list else 0

    # Log final summary
    summary_log = (
        f"\n=== Final Evaluation Metrics ===\n"
        f"Mean Recall@{k}: {mean_recall_at_k:.4f}\n"
        f"Mean MRR@{k}: {mean_reciprocal_rank_at_k:.4f}\n"
        f"Mean NDCG@{k}: {mean_ndcg_at_k:.4f}\n"
        f"=============================\n"
    )
    write_log(summary_log)

    # Create a DataFrame for better readability
    metrics_data = {
        "Metric": [f"Recall@{k}", f"Mean Reciprocal Rank@{k}", f"Normalized Discounted Cumulative Gain@{k}"],
        "Value": [mean_recall_at_k, mean_reciprocal_rank_at_k, mean_ndcg_at_k]
    }
    metrics_df = pd.DataFrame(metrics_data)
    
    print("\n=======>>> Evaluation Metrics Summary\n")
    print(metrics_df.to_string(index=False))

    return metrics_df  # Optionally return metrics for logging

## Loading Dataset

In [7]:
dataset_folder = "./dataset"  # Change this to your actual dataset path
train_dataset_path = f"{dataset_folder}/cleaned_acl_global_context_dataset_train.csv"
eval_dataset_path = f"{dataset_folder}/cleaned_acl_global_context_dataset_eval.csv"

In [12]:
pretrained_model_name_or_path = "facebook/bart-base"
max_token_limit = 400

# Initialize the config
config = BartConfig.from_pretrained(pretrained_model_name_or_path, attention_dropout=0.123)

# Initialize the tokenizer
tokenizer = BartTokenizer.from_pretrained(pretrained_model_name_or_path, truncation=True,
                                          padding='max_length', model_max_length=max_token_limit)

# Set up the model
model = BartForConditionalGeneration.from_pretrained(pretrained_model_name_or_path, config=config)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

cit_generation_config = GenerationConfig.from_model_config(model.config)

cit_generation_config.max_new_tokens = 25
cit_generation_config.do_sample = False
cit_generation_config.top_k = 50
cit_generation_config.num_return_sequences = 20
cit_generation_config.early_stopping = False
cit_generation_config.num_beams = 20
cit_generation_config.forced_bos_token_id = 0

cit_generation_config.num_beam_groups = 10
cit_generation_config.diversity_penalty = 1.5

In [24]:
# Example data to view dataset structure
"""data = {
        "train": [
            {"input": "Fill the mask with an appropriate citation: models are trained end-to-end using backpropagation
             and mini-batched Adam <mask> SGD. We use dropout regularization",
             "target": "Kingma and Ba, 2014"},
            # ...
        ],
        "validation": [
            {"input": "Fill the mask with an appropriate citation: The new policy is <mask>.",
             "target": "under review."},
            # ...
        ]
    }"""

train_dataset, eval_dataset = read_dataset()

data = {
    "train": train_dataset,
    "eval": eval_dataset
}

# Convert to Dataset
train_dataset = Dataset.from_pandas(pd.DataFrame(data["train"]))
validation_dataset = Dataset.from_pandas(pd.DataFrame(data["eval"]))

dataset = DatasetDict({
    "train": train_dataset,
    "eval": validation_dataset
})

# Preprocess the datasets
tokenized_datasets = dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/49584 [00:00<?, ? examples/s]

Map:   0%|          | 0/12591 [00:00<?, ? examples/s]

## Training Part

In [8]:
# Manually setting arguments (remove argparse)
custom_model_name = "citeBART_global_ACL200_rerun"
checkpoints_location = f"./checkpoints/{custom_model_name}"
model_save_location = f"./models/{custom_model_name}"

dataset_folder = "./dataset"  # Change this to your actual dataset path
train_dataset_path = f"{dataset_folder}/cleaned_acl_global_context_dataset_train.csv"
eval_dataset_path = f"{dataset_folder}/cleaned_acl_global_context_dataset_eval.csv"
all_citations_path = f"{dataset_folder}/cleaned_acl_global_citation_item_list.csv"

num_epochs = 15
warmup_steps = 500
train_and_eval_batch_sizes = 16
auto_find_batch_size_flag = True
skip_training = False

In [9]:
# Define log file location
log_file = os.path.join(model_save_location, "training_logs.txt")

# Function to log messages
def write_log(message):
    """Append messages to the log file."""
    with open(log_file, "a") as f:
        f.write(message + "\n")

# Identify the latest checkpoint if available
latest_checkpoint = None
if os.path.exists(checkpoints_location) and os.listdir(checkpoints_location):
    checkpoint_dirs = [
        os.path.join(checkpoints_location, d)
        for d in os.listdir(checkpoints_location)
        if "checkpoint" in d and os.path.isdir(os.path.join(checkpoints_location, d))
    ]
    
    if checkpoint_dirs:
        latest_checkpoint = max(checkpoint_dirs, key=os.path.getctime)

print("Latest Checkpoint:", latest_checkpoint)

Latest Checkpoint: ./checkpoints/citeBART_global_ACL200_rerun/checkpoint-92970


In [32]:
training_args = TrainingArguments(
    output_dir=checkpoints_location,
    overwrite_output_dir=True,
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=num_epochs,
    weight_decay=0.01,
    logging_strategy="epoch",
    warmup_steps=warmup_steps,
    save_strategy="epoch",
    save_total_limit=15,
    resume_from_checkpoint=latest_checkpoint is not None,
)

if auto_find_batch_size_flag:
    training_args.auto_find_batch_size = True
else:
    training_args.per_device_train_batch_size = train_and_eval_batch_sizes
    training_args.per_device_eval_batch_size = train_and_eval_batch_sizes

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["eval"],
    data_collator=data_collator,
    tokenizer=tokenizer
)

/opt/conda/lib/python3.12/site-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_2418/3286828109.py:21: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [17]:
import traceback

# Start training with error handling
try:
    if not skip_training:
        if latest_checkpoint:
            write_log(f"Resuming training from checkpoint: {latest_checkpoint}")
            train_output = trainer.train(resume_from_checkpoint=latest_checkpoint)
        else:
            write_log("Starting training from scratch...")
            train_output = trainer.train()

        trainer.save_model(model_save_location)
        tokenizer.save_pretrained(model_save_location)

        # Extract training metrics
        metrics = train_output.metrics
        metrics_table = pd.DataFrame([metrics])

        # Log metrics
        metrics_log = metrics_table.to_string(index=False)
        write_log("Training Completed Successfully ✅\n\nMetrics:\n" + metrics_log)

except Exception as e:
    error_message = f"Training Failed ❌\n\nError Details:\n{traceback.format_exc()}"
    
    # Log error
    write_log(error_message)

    # Re-raise the error so it doesn't silently fail
    raise

Epoch,Training Loss,Validation Loss
1,0.394500,0.021601
2,0.019700,0.015661
3,0.014700,0.013371
4,0.011800,0.011904
5,0.009700,0.011247
6,0.008200,0.010705
7,0.006900,0.010465
8,0.005900,0.010292
9,0.005000,0.010586
10,0.004300,0.010640


/opt/conda/lib/python3.12/site-packages/transformers/modeling_utils.py:2758: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


## Loading and Evaluating Model

In [33]:
import torch

# Load Model
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Define model directory path
model_path = "models/citeBART_global_ACL200_rerun"  # Adjust this if needed

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Load model
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)


trainer = Trainer(
    model=model,
    args=training_args, # Use the same training arguments as above (rerun it)
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["eval"],
    data_collator=data_collator,
    tokenizer=tokenizer
)

print("Model and tokenizer loaded successfully!")

Model and tokenizer loaded successfully!


/tmp/ipykernel_2418/4213989073.py:16: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [19]:
eval_results = trainer.evaluate()
print(f"\n*****************\n======>> Eval loss after fine-tuning: {eval_results['eval_loss']}\n"
      f"======>> Perplexity after fine-tuning: {math.exp(eval_results['eval_loss']):.2f}\n\n")


*****************
======>> Eval loss after fine-tuning: 0.011236312799155712
======>> Perplexity after fine-tuning: 1.01




In [21]:
# Run evaluation for epoch 15
calc_eval_metrics(eval_dataset)

100%|██████████| 12591/12591 [1:00:14<00:00,  3.48it/s]


=======>>> Evaluation Metrics Summary

                                  Metric    Value
                               Recall@10 0.680089
                 Mean Reciprocal Rank@10 0.509516
Normalized Discounted Cumulative Gain@10 0.550847


,Metric,Value
0,Recall@10,0.680089
1,Mean Reciprocal Rank@10,0.509516
2,Normalized Discounted Cumulative Gain@10,0.550847


### Understanding How the Tokenization Works

In [27]:
data["eval"][0]

{'masked_cit_context': 'A Hierarchy with, of, and for Preposition Supersenses </s> English prepositions are extremely frequent and extraordinarily polysemous. In some usages they contribute information about spatial, temporal, or causal roles/relations; in other cases they are institutionalized, somewhat arbitrarily, as case markers licensed by a particular governing verb, verb class, or syntactic construction. To facilitate automatic disambiguation, we propose a general-purpose, broadcoverage taxonomy of preposition functions that we call supersenses: these are coarse and unlexicalized so as to be tractable for efficient manual annotation, yet capture crucial semantic distinctions. Our resource, including extensive documentation of the supersenses, many example sentences, and mappings to other lexical resources, will be publicly released. Prepositions are perhaps the most beguiling yet pervasive lexicosyntactic class in English. They are everywhere; their functional versatility is diz

In [28]:
tokenized_datasets["eval"][0]

{'masked_cit_context': 'A Hierarchy with, of, and for Preposition Supersenses </s> English prepositions are extremely frequent and extraordinarily polysemous. In some usages they contribute information about spatial, temporal, or causal roles/relations; in other cases they are institutionalized, somewhat arbitrarily, as case markers licensed by a particular governing verb, verb class, or syntactic construction. To facilitate automatic disambiguation, we propose a general-purpose, broadcoverage taxonomy of preposition functions that we call supersenses: these are coarse and unlexicalized so as to be tractable for efficient manual annotation, yet capture crucial semantic distinctions. Our resource, including extensive documentation of the supersenses, many example sentences, and mappings to other lexical resources, will be publicly released. Prepositions are perhaps the most beguiling yet pervasive lexicosyntactic class in English. They are everywhere; their functional versatility is diz

In [45]:
def add_spaces_after_commas(text: str) -> str:
    return ', '.join(part.strip() for part in text.split(','))  #  NEW!!!!

# Read CSV and extract relevant column as a list (Optimized)

sentence = tokenized_datasets["eval"][0]["masked_cit_context"]

# Tokenize input
sentence = sentence.replace("<mask>", "<extra_id_0>", 1).replace("<mask>", "").replace("<extra_id_0>", "<mask>")

print(sentence)

input_ids = tokenizer.encode(sentence, return_tensors="pt", max_length=max_token_limit, truncation=True, padding="max_length").to("cuda")

print(input_ids)

# Generate outputs
outputs = model.generate(input_ids, generation_config=cit_generation_config)

print(outputs[0:4])

sample_decode = [tokenizer.decode(x) for x in outputs[0:4]]

print(sample_decode)

# Decode outputs efficiently
predictions = [add_spaces_after_commas(tokenizer.decode(output, skip_special_tokens=True).strip()) for output in outputs]

print(predictions[0:4])

A Hierarchy with, of, and for Preposition Supersenses </s> English prepositions are extremely frequent and extraordinarily polysemous. In some usages they contribute information about spatial, temporal, or causal roles/relations; in other cases they are institutionalized, somewhat arbitrarily, as case markers licensed by a particular governing verb, verb class, or syntactic construction. To facilitate automatic disambiguation, we propose a general-purpose, broadcoverage taxonomy of preposition functions that we call supersenses: these are coarse and unlexicalized so as to be tractable for efficient manual annotation, yet capture crucial semantic distinctions. Our resource, including extensive documentation of the supersenses, many example sentences, and mappings to other lexical resources, will be publicly released. Prepositions are perhaps the most beguiling yet pervasive lexicosyntactic class in English. They are everywhere; their functional versatility is dizzying and largely idiosy

tensor([[    2,     0,   387,  4218,  4400,  1076,   482,  6708,     2,     1,
             1,     1,     1],
        [    2,     0,   534,  9683, 13184,     8, 16816,  2001, 18708,     6,
          5241,     2,     1],
        [    2,     0, 26519,  7180,     8, 18485,  2186,     6,  2338,     2,
             1,     1,     1],
        [    2,     0, 26519, 25383,   405,  4400,  1076,   482,  1824,     2,
             1,     1,     1]], device='cuda:0')
['</s><s>Baker et al., 1998</s><pad><pad><pad><pad>', '</s><s>Gildea and Jurafsky, 2002</s><pad>', '</s><s>Brody and Lapata, 2009</s><pad><pad><pad>', '</s><s>Broscheit et al., 2010</s><pad><pad><pad>']
['Baker et al., 1998', 'Gildea and Jurafsky, 2002', 'Brody and Lapata, 2009', 'Broscheit et al., 2010']


The tokenization here appears to be done per common tokens rather than per word. This means that instead of splitting at every space or punctuation mark, the text is broken into subword units that are frequently occurring in the model’s vocabulary.

Why is it not per-word tokenization?
1.	Some numbers and names are single tokens:
	- For example, “1998” or “Baker” might be mapped to a single token if they are common in the training corpus.
	- If it were per-word tokenization, each word would correspond to exactly one token, which is unlikely in this case.
2.	Presence of subword units:
	- Words like “et al.” or “Jurafsky” might be split into multiple tokens if they are not in the vocabulary as a single unit.
	- The token IDs suggest that longer words (like “Brody and Lapata, 2009”) are composed of multiple subword tokens.
3.	Padding and special tokens:
	- The sequences include "<s>" (start token), "</s>" (end token), and <pad> (padding), which are typically used in models like BERT, T5, or GPT variants.
	- These models use subword tokenization methods like Byte-Pair Encoding (BPE) or SentencePiece (used in T5, mT5, and some BART models).
